In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [41]:
transform = transforms.Compose([
    transforms.ToTensor(),#转换为Tensor格式、
    transforms.Normalize((0.5,), (0.5,))#标准化处理，均值为0.5，标准差为0.5
])

# 使用 datasets.FashionMNIST 而不是 ImageFolder，因为数据是 idx 格式的二进制文件
# root 指向包含数据文件的父目录
train_dataset = datasets.FashionMNIST(root='.\data', train=True, download=False, transform=transform)
test_dataset = datasets.FashionMNIST(root='.\data', train=False, download=False, transform=transform)
# 取出第一条数据
img, label = train_dataset[0]

print(f"数据类型: {type(img)}")
print(f"图像形状: {img.shape}")
print(f"标签值: {label}")
print(f"像素值范围: {img.min().item():.2f} 到 {img.max().item():.2f}")

# 创建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

数据类型: <class 'torch.Tensor'>
图像形状: torch.Size([1, 28, 28])
标签值: 9
像素值范围: -1.00 到 1.00


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # conv1:
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1,
            stride=1
        )
        self.relu = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 14 * 14, 128)
        self.fc2 = nn.Linear(128,10)


        # conv2:

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
        
model = CNN()

In [43]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CNN(
  (conv1): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=1960, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [44]:
def train(model, device, train_loader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        for batch_idx, (x, y) in enumerate(train_loader):
            data, target = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 1 == 0:
            print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")
    

In [45]:
def test_model(model, device, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            data, target = x.to(device), y.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            # 参数解析：
            #   outputs.data: 模型输出的张量。
            #   1: 指定在维度 1（即 10 个类别的方向）上取最大值。
            # 操作：寻找每一行中分数最大的那个值的索引。
            # 【返回结果维度】:
            #   _: [64] (最大值本身，在这里我们不需要，所以用下划线忽略)
            #   predicted: [64] (最大值所在的索引，即预测的类别)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [46]:
epochs = int(input('请输入训练轮数：'))

train(model, device, train_loader, optimizer, criterion, epochs)
test_model(model, device, test_loader)

Epoch [1/10], Loss: 0.4786
Epoch [2/10], Loss: 0.1162
Epoch [3/10], Loss: 0.2701
Epoch [4/10], Loss: 0.2959
Epoch [5/10], Loss: 0.4552
Epoch [6/10], Loss: 0.1028
Epoch [7/10], Loss: 0.0593
Epoch [8/10], Loss: 0.2890
Epoch [9/10], Loss: 0.1330
Epoch [10/10], Loss: 0.2297
Test Accuracy: 90.30%
